# Airline passenger satisfaction

**Playground Series S6E10.** Predict `satisfaction` (True/False) for airline passengers from 21 features. The metric is ROC AUC.

**Workflow.** Every experiment uses the fixed stratified folds in `folds_seed42.csv`. Each run has a name (`CFG.EXPERIMENT`), appends its scores to `experiments.csv`, saves its predictions in `preds/` and writes its own submission.

**Models.**
- LightGBM runs on the CPU.
- XGBoost and CatBoost run on the CPU, or on an NVIDIA GPU if one is present.
- The neural net runs on the Apple GPU (MPS) on a Mac.

`CFG.REUSE` loads a model's saved predictions from an earlier experiment instead of retraining it. That matters most for the tuned 5-seed LightGBM, which takes about 35 minutes to train.

Run top to bottom from the project root or from `notebooks/`. Inputs are read from `data/`; the log, predictions and submissions go to `results/`.

In [ ]:
import copy  # Snapshot the best neural-net weights.
import gc  # Release temporary fold objects after each fit.
import math  # Count neural-net batches.
import random  # Seed Python's random number generator.
import sys  # Detect macOS.
import time  # Measure cross-validation runtime.
from pathlib import Path  # Work with local and Kaggle paths.

import numpy as np  # Store predictions and do numerical calculations.
import pandas as pd  # Load CSV files and engineer tabular features.
import lightgbm as lgb  # Gradient-boosted trees (CPU).
import xgboost as xgb  # Gradient-boosted trees (GPU when available).
import torch  # Neural net (Apple GPU via MPS on a Mac).
import torch.nn as nn  # Neural-net layers.
import torch.nn.functional as F  # Neural-net loss.
from catboost import CatBoostClassifier, Pool  # Gradient-boosted trees (GPU when available).
from sklearn.metrics import log_loss, roc_auc_score  # Evaluate predictions.
from sklearn.model_selection import StratifiedKFold  # Preserve class balance in folds.

if sys.platform == "darwin":  # On a Mac, PyTorch and LightGBM/XGBoost load separate OpenMP runtimes that can deadlock.
    torch.set_num_threads(1)


class CFG:
    EXPERIMENT = "exp13_nn_tuned_cb_orig"  # Name for experiments.csv, preds/, and the submission copy.
    BASELINE = "exp12_seeds"  # Experiment that deltas are measured against.
    DEBUG = False  # Set True to check the workflow quickly on 20,000 rows (ignores REUSE).
    FAST = False  # Set True to screen an idea with LightGBM only at a higher learning rate (ignores REUSE).
    GPU = torch.cuda.is_available()  # XGBoost and CatBoost need an NVIDIA GPU; on a Mac they use the CPU.
    NN_DEVICE = "cuda" if GPU else "mps" if torch.backends.mps.is_available() else "cpu"  # Apple GPU on a Mac.
    RUN_LIGHTGBM = True  # Which models go into the blend.
    RUN_XGBOOST = True
    RUN_CATBOOST = True  # With CB_ALL_CAT it adds +0.0004 to the exp09 blend (screen_catboost_cat.py, folds 1-2).
    RUN_CATBOOST_FAST = True  # A second CatBoost member: CB_FAST_SEEDS averaged at CB_FAST_LR, blended beside CatBoost.
    RUN_CATBOOST_ORIG = True  # Third CatBoost member: raw columns + the original rows (flagged) in every training fold.
    RUN_NN = True  # Neural net for blend diversity.
    # Reuse saved predictions instead of retraining: {model: experiment}. Only valid when that experiment used the same
    # settings, features and folds; the files must be in preds/. Set a model to None to retrain it in this run.
    REUSE = {"LightGBM": "exp09_orig_lookup", "XGBoost": "exp09_orig_lookup", "CatBoost": "exp10_catboost_cat",
             "CatBoostFast": "exp12_seeds", "CatBoostOrig": None, "NN": None}
    REFIT = True  # After the blend, retrain the blended models on all training rows and submit their test predictions.
    REFIT_TREE_FACTOR = 1.15  # Refit tree count = mean best iteration across folds x this (25% more training rows).
    REFIT_LGB_TREES = 2148  # Used only when LightGBM is reused without saved refit predictions (exp09's size).
    REFIT_NN_EPOCHS = 14  # Used only when the NN is reused; a run that trains it uses its median best epoch.
    # A reused model's refit loads that experiment's saved refit predictions when they exist.
    SEED = 42  # Use the same random seed in splitting and training.
    N_FOLDS = 5  # Train and validate on five fixed partitions.
    METRIC = "auc"  # Change to "logloss" if the competition metric differs.
    TARGET = "satisfaction"  # Name of the label in train.csv.
    ID = "id"  # Key used to restore the submission's row order.
    EARLY_STOP = 200  # Stop when validation quality stops improving.
    LGB_ITERS = 10000  # Maximum LightGBM trees before early stopping.
    LGB_LR = 0.02  # LightGBM learning rate.
    LGB_SEEDS = 5  # LightGBM is averaged over this many training seeds (same folds).
    LGB_PARAMS = dict(  # Best of 45 wider Optuna trials (tune_lgb2.py, lr 0.1 on folds 0-2).
        num_leaves=80, min_child_samples=7, subsample=0.9557, colsample_bytree=0.5715, reg_lambda=0.1676,
        reg_alpha=0.4640, min_split_gain=0.3337, max_bin=1023, cat_smooth=16.41, cat_l2=2.798, path_smooth=12.14)
    XGB_ITERS = 5000  # Maximum XGBoost trees before early stopping.
    XGB_LR = 0.05  # XGBoost learning rate.
    XGB_SEEDS = 3  # XGBoost is averaged over this many training seeds (same folds).
    XGB_PARAMS = dict(  # Best of 25 Optuna trials (tune_xgb.py, lr 0.1 on folds 0-2).
        max_depth=12, min_child_weight=13.5446, subsample=0.8902, colsample_bytree=0.7513, reg_lambda=17.9858,
        reg_alpha=5.5249, gamma=0.3935, max_bin=512)
    CB_ITERS = 6000  # Maximum CatBoost trees before early stopping.
    CB_LR = 0.06  # CatBoost learning rate (the screen used 0.15).
    CB_DEPTH = 7  # CatBoost tree depth (depth 6/8, l2 1/10 and other settings screened no better).
    CB_FAST_LR = 0.15  # CatBoostFast learning rate: ~4x faster than CB_LR for ~0.0001 per model.
    CB_FAST_SEEDS = [1, 2]  # CatBoostFast seed offsets; offset 0 is the main CatBoost member.
    CB_ORIG_LR = 0.15  # CatBoostOrig learning rate (screen_catboost_orig.py: +0.0001 blend on folds 1-2).
    CB_ORIG_SEEDS = [0]  # CatBoostOrig seed offsets.
    CB_ALL_CAT = True  # Treat every raw column as categorical, so CatBoost target-encodes values and their combinations.
    NN_SEEDS = 5  # The neural net is averaged over this many training seeds.
    # NN settings above: best of 20 Optuna trials (tune_nn.py, folds 1-2): 0.95970 vs 0.95883 before.
    NN_EPOCHS = 25  # Fixed one-cycle schedule; the best epoch on the validation fold is kept.
    NN_BATCH = 1024  # Rows per gradient step.
    NN_LR = 2.1e-3  # Peak learning rate of the one-cycle schedule.
    NN_WEIGHT_DECAY = 3.6e-4  # AdamW weight decay.
    NN_HIDDEN = (256, 128)  # Widths of the hidden layers.
    NN_DROPOUT = 0.27  # Dropout after each hidden layer.
    NN_EMB_DIM = 8  # Embedding size for columns with up to 10 values (categories and ratings).
    NN_EMB_DIM_LARGE = 4  # Embedding size for columns with more values (the binned columns).
    NN_BINS = True  # Also give binned versions of these numeric columns their own embeddings.
    NN_BIN_COLS = {"Age": None, "Flight Distance": None, "Departure Delay in Minutes": None,
                   "Arrival Delay in Minutes": None}  # None: one embedding per integer value; a number: that many quantile bins.
    CAT_COLS = ["Gender", "Customer Type", "Type of Travel", "Class"]  # Text categories.
    RATING_COLS = ["Inflight wifi service", "Departure/Arrival time convenient", "Ease of Online booking", "Gate location",
                   "Food and drink", "Online boarding", "Seat comfort", "Inflight entertainment", "On-board service",
                   "Leg room service", "Baggage handling", "Checkin service", "Cleanliness"]  # 0-5 survey ratings.
    FE_RATINGS = False  # Row-wise summaries of the ratings (sum, mean, std, counts of 0/low/5).
    FE_DELAY = False  # Delay difference, logs, and a missing-arrival-delay flag.
    FE_BINS = False  # Age decade and log flight distance.
    FE_INTERACT = False  # Class x Type of Travel as an extra categorical.
    FE_ORIGINAL = True  # Original-dataset satisfaction rates: one per column, the rating profile, the category combination.
    ORIGINAL_SMOOTH = 20  # Pseudo-count pulling rare original matches toward the original's overall rate.
    USE_ORIGINAL = False  # Append the original dataset to every training fold (never to validation).
    ORIGINAL_PATH = "original/original_all.csv"  # Inside data/: the original dataset (train + test combined).
    FE_ROUTE = False  # Route profile: per Flight Distance value, the mean of every other column and its row count (train + test, no labels).
    FE_COUNTS = False  # Value counts of every column's value and Flight Distance // 10, // 100, % 100, % 1000 and digit count.
    FE_AUX = []  # From results/features/aux_ev.parquet (scripts/build_features.py): "ev" expected ratings, "res" rating minus expected.
    UPDATE_SUBMISSION = True  # Also overwrite submission.csv; False writes only the submission_<EXPERIMENT> copies.


if CFG.FE_INTERACT:  # The interaction is encoded like the other categories.
    CFG.CAT_COLS = CFG.CAT_COLS + ["Class_Travel"]
if CFG.DEBUG:  # Shorten every model when testing code changes; saved predictions would not match the sample.
    CFG.LGB_ITERS, CFG.CB_ITERS, CFG.XGB_ITERS, CFG.EARLY_STOP = 300, 300, 300, 50
    CFG.LGB_SEEDS = CFG.XGB_SEEDS = CFG.NN_SEEDS = 1
    CFG.NN_EPOCHS, CFG.REUSE = 3, {}
    CFG.REFIT_LGB_TREES, CFG.REFIT_NN_EPOCHS = min(CFG.REFIT_LGB_TREES, 300), min(CFG.REFIT_NN_EPOCHS, 3)
if CFG.FAST:  # FAST scores are only comparable with other FAST runs.
    CFG.RUN_XGBOOST = CFG.RUN_CATBOOST = CFG.RUN_CATBOOST_FAST = CFG.RUN_CATBOOST_ORIG = CFG.RUN_NN = False
    CFG.LGB_LR, CFG.LGB_SEEDS, CFG.REUSE = 0.1, 1, {}

assert CFG.REFIT_NN_EPOCHS <= CFG.NN_EPOCHS, "The refit stops inside the NN_EPOCHS schedule."
random.seed(CFG.SEED)  # Make Python-side random choices repeatable.
np.random.seed(CFG.SEED)  # Make NumPy-side random choices repeatable.
torch.manual_seed(CFG.SEED)  # Make PyTorch-side random choices repeatable.
ROOT = Path.cwd() if (Path.cwd() / "data").is_dir() else Path.cwd().parent  # Project root: run from it or from notebooks/.
DATA_DIR = ROOT / "data"  # Competition files, folds, the original dataset and the public pool.
ON_KAGGLE = Path("/kaggle/working").exists()
if ON_KAGGLE and not (DATA_DIR / "train.csv").is_file():  # On Kaggle, find the mounted competition.
    DATA_DIR = sorted(Path("/kaggle/input").rglob("train.csv"))[0].parent
OUT_DIR = Path("/kaggle/working") if ON_KAGGLE else ROOT / "results"  # experiments.csv, preds/, submissions/.
PRED_DIR = OUT_DIR / "preds"  # Saved out-of-fold and test predictions, one pair per model and experiment.
SUB_DIR = OUT_DIR if ON_KAGGLE else OUT_DIR / "submissions"  # submission.csv (the file to upload) and per-experiment copies.
FOLD_DIR = OUT_DIR if ON_KAGGLE else DATA_DIR  # Where the fixed folds are saved.
SUB_DIR.mkdir(parents=True, exist_ok=True)


def score(y_true, probability):
    """Return the selected competition metric for probability predictions."""
    if CFG.METRIC == "auc":  # Higher AUC means better ranking.
        return roc_auc_score(y_true, probability)
    return log_loss(y_true, probability)  # Lower log loss means better calibration.


print(f"Experiment: {CFG.EXPERIMENT} | Input: {DATA_DIR} | Output: {OUT_DIR} | NN device: {CFG.NN_DEVICE} | Debug: {CFG.DEBUG} | Fast: {CFG.FAST}")
print("Reusing:", {name: source for name, source in CFG.REUSE.items() if source} or "nothing")

## 2. Load and check the data

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")  # Labelled training rows.
test = pd.read_csv(DATA_DIR / "test.csv")  # Rows that need predictions.
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")  # Required output layout.

assert CFG.TARGET in train.columns, f"Missing target: {CFG.TARGET}"  # Confirm the label exists.
assert set(train.columns) - {CFG.TARGET} == set(test.columns), "Train/test columns differ."  # Matching inputs.
assert sample[CFG.ID].is_unique and set(sample[CFG.ID]) == set(test[CFG.ID]), "Submission IDs differ from test IDs."

train[CFG.TARGET] = train[CFG.TARGET].astype(str).str.strip().str.lower().map({"true": 1, "false": 0, "1": 1, "0": 0})  # True/False to 1/0.
assert train[CFG.TARGET].isin([0, 1]).all(), "The target must contain only 0 and 1."  # Reject invalid labels.
train[CFG.TARGET] = train[CFG.TARGET].astype("int8")  # Store the label compactly.

original = None  # Optional extra labelled rows, used only for training.
if CFG.USE_ORIGINAL:
    assert CFG.ORIGINAL_PATH and (DATA_DIR / CFG.ORIGINAL_PATH).is_file(), "Set CFG.ORIGINAL_PATH to the original dataset CSV."
    original = pd.read_csv(DATA_DIR / CFG.ORIGINAL_PATH)
    original = original.drop(columns=[c for c in original.columns if c.lower().startswith("unnamed")] + [c for c in ["id"] if c in original.columns])
    original[CFG.TARGET] = original[CFG.TARGET].astype(str).str.strip().str.lower().isin(["satisfied", "true", "1"]).astype("int8")
    missing = set(test.columns) - {CFG.ID} - set(original.columns)
    assert not missing, f"Original dataset lacks columns: {missing}"
    original = original[[c for c in test.columns if c != CFG.ID] + [CFG.TARGET]].reset_index(drop=True)

lookup_source = None  # Labelled original rows behind the FE_ORIGINAL lookups; never appended to training.
if CFG.FE_ORIGINAL:
    assert (DATA_DIR / CFG.ORIGINAL_PATH).is_file(), "Download the original dataset to CFG.ORIGINAL_PATH first."
    lookup_source = pd.read_csv(DATA_DIR / CFG.ORIGINAL_PATH)
    lookup_source[CFG.TARGET] = lookup_source[CFG.TARGET].astype(str).str.strip().str.lower().isin(["satisfied", "true", "1"]).astype("int8")

if CFG.DEBUG:  # Use a reproducible subset for a quick workflow check.
    train = train.sample(n=min(20_000, len(train)), random_state=CFG.SEED).reset_index(drop=True)

print("train:", train.shape, "test:", test.shape, "sample:", sample.shape)  # Check data sizes.
print("missing cells:", train.isna().sum()[lambda s: s > 0].to_dict(), test.isna().sum()[lambda s: s > 0].to_dict())  # Only Arrival Delay.
if original is not None:
    print("original:", original.shape, "positive rate:", round(original[CFG.TARGET].mean(), 4))
print("positive rate:", round(train[CFG.TARGET].mean(), 4))  # Check class balance.

## 3. Features

Each block of engineered columns sits behind a `CFG.FE_*` switch. The exp01 switches are off, because they lowered the score. `FE_ORIGINAL` is on: it adds the original dataset's satisfaction rate for each column's value, for the full rating-and-category profile (with a match count), and for the category combination.

In [ ]:
BASE_COLS = [name for name in test.columns if name != CFG.ID]  # The 21 raw features.


def original_rate(frame, cols):
    """Smoothed satisfaction rate, and match count, of original-dataset rows sharing `cols` with each row.

    The labels come only from the original dataset, so these columns cannot leak across the competition folds.
    """
    def key(table):
        return table[cols[0]].fillna(-1) if len(cols) == 1 else table[cols].fillna(-1).astype(str).agg("|".join, axis=1)

    stats = lookup_source.groupby(key(lookup_source))[CFG.TARGET].agg(["sum", "count"])
    keys, prior = key(frame), lookup_source[CFG.TARGET].mean()
    total, count = keys.map(stats["sum"]).fillna(0), keys.map(stats["count"]).fillna(0)
    return ((total + prior * CFG.ORIGINAL_SMOOTH) / (count + CFG.ORIGINAL_SMOOTH)).astype("float32"), count.astype("float32")


if CFG.FE_ROUTE or CFG.FE_COUNTS:  # Label-free statistics over the train and test feature rows (screen_aux_route.py).
    both = pd.concat([train[BASE_COLS], test[BASE_COLS]], ignore_index=True)
    for name in CFG.CAT_COLS[:4]:
        both[name] = both[name].astype("category").cat.codes
    route_stats = {f"route_mean_{name}": both.groupby("Flight Distance")[name].mean() for name in BASE_COLS if name != "Flight Distance"}
    route_stats["route_n"] = both.groupby("Flight Distance").size()
    value_counts = {name: both[name].value_counts() for name in ["Age", "Departure Delay in Minutes", "Arrival Delay in Minutes"]}
if CFG.FE_AUX:  # Expected value of each rating given the other 20 columns, cross-fitted over train + test (no labels).
    assert not CFG.USE_ORIGINAL, "Original rows have no auxiliary-task values."
    aux_table = pd.read_parquet(ROOT / "results" / "features" / "aux_ev.parquet")


def make_features(frame):
    """Return the model inputs for one table, adding the engineered columns that CFG enables."""
    out = frame.copy()
    if CFG.FE_RATINGS:
        ratings = out[CFG.RATING_COLS]
        out["rating_sum"], out["rating_mean"], out["rating_std"] = ratings.sum(axis=1), ratings.mean(axis=1), ratings.std(axis=1)
        out["rating_min"], out["rating_max"] = ratings.min(axis=1), ratings.max(axis=1)
        out["n_zero"], out["n_low"], out["n_five"] = (ratings == 0).sum(axis=1), ((ratings >= 1) & (ratings <= 2)).sum(axis=1), (ratings == 5).sum(axis=1)
    if CFG.FE_DELAY:
        dep, arr = out["Departure Delay in Minutes"], out["Arrival Delay in Minutes"]
        out["delay_gap"], out["arrival_missing"] = arr - dep, arr.isna().astype("int8")
        out["log_dep_delay"], out["log_arr_delay"] = np.log1p(dep), np.log1p(arr)
    if CFG.FE_BINS:
        out["age_decade"], out["log_distance"] = out["Age"] // 10, np.log1p(out["Flight Distance"])
    if CFG.FE_ORIGINAL:  # Screened in screen_orig_features.py: +0.00034 LightGBM OOF, 5/5 folds.
        for name in BASE_COLS:
            out[f"orig_te_{name}"] = original_rate(frame, [name])[0]
        out["orig_te_profile"], out["orig_n_profile"] = original_rate(frame, CFG.RATING_COLS + CFG.CAT_COLS[:4])
        out["orig_te_categories"] = original_rate(frame, CFG.CAT_COLS[:4])[0]
    if CFG.FE_ROUTE:
        for name, values in route_stats.items():
            out[name] = frame["Flight Distance"].map(values).astype("float32")
    if CFG.FE_COUNTS:
        for name, counts in value_counts.items():
            out[f"count_{name}"] = frame[name].map(counts).fillna(0).astype("float32")
        distance = frame["Flight Distance"]
        out["fd_div10"], out["fd_div100"] = (distance // 10).astype("float32"), (distance // 100).astype("float32")
        out["fd_mod100"], out["fd_mod1000"] = (distance % 100).astype("float32"), (distance % 1000).astype("float32")
        out["fd_digits"] = distance.astype(str).str.len().astype("float32")
    if CFG.FE_AUX:
        expected = frame[[CFG.ID]].merge(aux_table, on=CFG.ID, how="left")
        assert expected.notna().all().all(), "aux_ev.parquet lacks some ids: rerun scripts/build_features.py aux."
        for name in CFG.RATING_COLS:
            if "ev" in CFG.FE_AUX:
                out[f"aux_ev_{name}"] = expected[f"aux_ev_{name}"].to_numpy()
            if "res" in CFG.FE_AUX:
                out[f"aux_res_{name}"] = (frame[name].to_numpy() - expected[f"aux_ev_{name}"].to_numpy()).astype("float32")
    if CFG.FE_INTERACT:
        out["Class_Travel"] = out["Class"].astype(str) + "|" + out["Type of Travel"].astype(str)
    return out


train = make_features(train)  # Engineer the training features.
test = make_features(test)  # Engineer the identical test features.
if original is not None:
    original = make_features(original)
FEATURES = [name for name in test.columns if name != CFG.ID]  # Exclude the ID from modeling.

for name in CFG.CAT_COLS:  # Give every table exactly the same category codes.
    tables = [train, test] + ([original] if original is not None else [])
    vocabulary = sorted(set().union(*(set(t[name].astype(str)) for t in tables)))  # Label-free vocabulary.
    for t in tables:
        t[name] = pd.Categorical(t[name].astype(str), categories=vocabulary)

print(f"{len(FEATURES)} features: {len(CFG.CAT_COLS)} categorical, {len(FEATURES) - len(CFG.CAT_COLS)} numeric")

## 4. Fixed folds

In [ ]:
y = train[CFG.TARGET].to_numpy()  # Keep labels as a compact NumPy array.
fold_path = FOLD_DIR / f"folds_seed{CFG.SEED}.csv"  # Saved fold IDs keep every experiment on identical splits.
if fold_path.is_file() and not CFG.DEBUG:  # Reuse the saved assignment.
    saved_folds = pd.read_csv(fold_path)
    assert saved_folds[CFG.ID].equals(train[CFG.ID]), "Saved folds do not match train.csv."
    fold_id = saved_folds["fold"].to_numpy().astype("int8")
    print(f"Loaded folds from {fold_path.name}")
else:  # Build the stratified assignment once.
    assert not any(CFG.REUSE.values()), f"Reused predictions need the {fold_path.name} they were made with; it is missing or does not match."
    fold_id = np.full(len(train), -1, dtype="int8")
    for number, (_, valid_idx) in enumerate(StratifiedKFold(CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED).split(train, y)):
        fold_id[valid_idx] = number
    if not CFG.DEBUG:  # A debug sample must not overwrite the full assignment.
        pd.DataFrame({CFG.ID: train[CFG.ID], "fold": fold_id}).to_csv(fold_path, index=False)
assert (fold_id >= 0).all(), "Every training row needs a fold."

## 5. Models

In [ ]:
def make_lightgbm(seed, n_estimators):
    """LightGBM classifier with the tuned settings."""
    return lgb.LGBMClassifier(n_estimators=n_estimators, learning_rate=CFG.LGB_LR, subsample_freq=1,
                              random_state=CFG.SEED + seed, verbose=-1, **CFG.LGB_PARAMS)


def fit_lightgbm(x_train, y_train, x_valid, y_valid, x_test):
    """Fit one LightGBM fold per seed on the CPU and return seed-averaged validation and test probabilities."""
    valid_pred, test_pred, best_iterations = 0.0, 0.0, []
    for seed in range(CFG.LGB_SEEDS):
        model = make_lightgbm(seed, CFG.LGB_ITERS)
        model.fit(x_train, y_train, eval_set=[(x_valid, y_valid)], eval_metric="auc" if CFG.METRIC == "auc" else "binary_logloss",
                  callbacks=[lgb.early_stopping(CFG.EARLY_STOP, verbose=False)])  # Stop using validation data.
        best_iterations.append(model.best_iteration_)
        fit_lightgbm.best_iterations.append(model.best_iteration_)  # All folds, for the refit.
        valid_pred += model.predict_proba(x_valid, num_iteration=model.best_iteration_)[:, 1] / CFG.LGB_SEEDS
        test_pred += model.predict_proba(x_test, num_iteration=model.best_iteration_)[:, 1] / CFG.LGB_SEEDS
    print(f"  LightGBM best iterations per seed: {best_iterations}")  # Their mean x1.15 sets REFIT_LGB_TREES.
    fit_lightgbm.last_model = model  # Kept for feature importance.
    return valid_pred, test_pred


fit_lightgbm.best_iterations = []


def fit_xgboost(x_train, y_train, x_valid, y_valid, x_test):
    """Fit one XGBoost fold per seed (GPU when available) and return seed-averaged validation and test probabilities."""
    valid_pred, test_pred = 0.0, 0.0
    for seed in range(CFG.XGB_SEEDS):
        model = xgb.XGBClassifier(
            n_estimators=CFG.XGB_ITERS, learning_rate=CFG.XGB_LR, tree_method="hist", enable_categorical=True,
            device="cuda" if CFG.GPU else "cpu", eval_metric="auc" if CFG.METRIC == "auc" else "logloss",
            early_stopping_rounds=CFG.EARLY_STOP, random_state=CFG.SEED + seed, n_jobs=-1, **CFG.XGB_PARAMS,
        )
        model.fit(x_train, y_train, eval_set=[(x_valid, y_valid)], verbose=False)
        stop = (0, model.best_iteration + 1)
        valid_pred += model.predict_proba(x_valid, iteration_range=stop)[:, 1] / CFG.XGB_SEEDS
        test_pred += model.predict_proba(x_test, iteration_range=stop)[:, 1] / CFG.XGB_SEEDS
    return valid_pred, test_pred


CB_CAT_FEATURES = BASE_COLS if CFG.CB_ALL_CAT else CFG.CAT_COLS  # Columns CatBoost treats as categorical.


def catboost_inputs(frame):
    """CatBoost input: categorical columns as strings (all raw columns with CB_ALL_CAT), the rest numeric."""
    return frame.assign(**{name: frame[name].astype(str) for name in CB_CAT_FEATURES})


def make_catboost(iterations, early_stopping=True, learning_rate=None, seed=0):
    """CatBoost classifier with the configured settings (GPU when available)."""
    return CatBoostClassifier(
        iterations=iterations, learning_rate=learning_rate or CFG.CB_LR, depth=CFG.CB_DEPTH, l2_leaf_reg=3,
        loss_function="Logloss", eval_metric="AUC" if CFG.METRIC == "auc" else "Logloss",
        od_type="Iter" if early_stopping else None, od_wait=CFG.EARLY_STOP if early_stopping else None,
        use_best_model=early_stopping, random_seed=CFG.SEED + seed, verbose=False, thread_count=-1,
        allow_writing_files=False, task_type="GPU" if CFG.GPU else "CPU",
    )


def catboost_fitter(learning_rate, seeds, prepare=catboost_inputs, cat_features=CB_CAT_FEATURES, extra=None):
    """Fit function for one CatBoost blend member: seeds averaged; best iterations kept for its refit.

    `prepare` turns feature frames into this member's inputs; `extra` (inputs, labels) rows join every training set.
    """
    def fit(x_train, y_train, x_valid, y_valid, x_test):
        valid_pred, test_pred, iterations = 0.0, 0.0, []
        if extra is not None:
            x_train, y_train = pd.concat([x_train, extra[0]], ignore_index=True), np.concatenate([y_train, extra[1]])
        for seed in seeds:
            model = make_catboost(CFG.CB_ITERS, learning_rate=learning_rate, seed=seed)
            model.fit(Pool(x_train, y_train, cat_features=cat_features),
                      eval_set=Pool(x_valid, y_valid, cat_features=cat_features))
            iterations.append(model.get_best_iteration() + 1)
            valid_pred += model.predict_proba(x_valid)[:, 1] / len(seeds)
            test_pred += model.predict_proba(x_test)[:, 1] / len(seeds)
        fit.best_iterations.extend(iterations)  # All folds and seeds, for the refit.
        print(f"  CatBoost (lr {learning_rate}) best iterations per seed: {iterations}")
        return valid_pred, test_pred

    fit.best_iterations, fit.learning_rate, fit.seeds = [], learning_rate, seeds
    fit.prepare, fit.cat_features, fit.extra = prepare, cat_features, extra
    return fit


fit_catboost = catboost_fitter(CFG.CB_LR, [0])  # The main member (exp10).
fit_catboost_fast = catboost_fitter(CFG.CB_FAST_LR, CFG.CB_FAST_SEEDS)  # Extra seeds at the faster rate.


def catboost_orig_inputs(frame):
    """CatBoostOrig input: the raw columns as categories plus an is_original flag (0 for competition rows).

    No lookup columns: for original rows they would be built from those rows' own labels.
    """
    return frame[BASE_COLS].astype(str).assign(is_original=0)


ORIGINAL_ROWS = None  # The original dataset as CatBoostOrig training rows.
if CFG.RUN_CATBOOST_ORIG and lookup_source is not None:
    ORIGINAL_ROWS = (lookup_source[BASE_COLS].astype(str).assign(is_original=1), lookup_source[CFG.TARGET].to_numpy())
fit_catboost_orig = catboost_fitter(CFG.CB_ORIG_LR, CFG.CB_ORIG_SEEDS, prepare=catboost_orig_inputs,
                                    cat_features=BASE_COLS, extra=ORIGINAL_ROWS)
CATBOOST_FITTERS = {"CatBoost": fit_catboost, "CatBoostFast": fit_catboost_fast, "CatBoostOrig": fit_catboost_orig}


### Neural net

- **Embeddings.** Categories and the 0-5 ratings each get a small embedding. With `NN_BINS`, Age, Flight Distance and the two delays get one embedding per exact value (exp11). Quantile bins were cruder: most delays are 0, so 32 bins collapsed to 5. That lets the net learn non-linear shapes in those columns, which it otherwise finds hard.
- **Numeric inputs.** The same columns, and the ratings, are also fed in as scaled numbers. Skewed columns are log-transformed, filled with the training median, and standardised with statistics from the training table only, which uses no labels.
- **Training.** Each seed trains for a fixed one-cycle schedule, and the epoch with the best validation score is kept, like the trees' early stopping. The run prints the best epoch for each seed. If those are mostly the last epochs, more epochs may help.

In [ ]:
NN_LOG_COLS = (["Flight Distance", "Departure Delay in Minutes", "Arrival Delay in Minutes"]  # Long right tails.
               + [name for name in FEATURES if name.startswith(("orig_n_", "route_n", "count_"))])
NN_EMBED = [name for name in CFG.CAT_COLS + CFG.RATING_COLS if name in FEATURES]  # Columns given embeddings.
NN_NUMERIC = [name for name in FEATURES if name not in CFG.CAT_COLS]  # Ratings appear here too, as numbers.


def nn_numeric(frame):
    """Numeric inputs with log transforms applied to the skewed columns."""
    out = frame[NN_NUMERIC].astype("float32")
    for name in NN_LOG_COLS:
        if name in out:
            out[name] = np.log1p(out[name].clip(lower=0))
    return out


reference = nn_numeric(train[FEATURES])  # Scaling statistics come from the training table only.
nn_fill = reference.median()
nn_mean, nn_std = reference.fillna(nn_fill).mean(), reference.fillna(nn_fill).std().replace(0, 1)

# Binned columns: quantile edges from the training table, or None for one embedding per integer value.
nn_bin_edges = {name: None if bins is None else np.unique(np.nanquantile(train[name], np.linspace(0, 1, bins + 1)[1:-1]))
                for name, bins in CFG.NN_BIN_COLS.items() if CFG.NN_BINS and name in FEATURES}
nn_int_max = {name: int(max(train[name].max(), test[name].max())) for name, edges in nn_bin_edges.items() if edges is None}


def bin_codes(values, name):
    """Embedding index for one binned column; index 0 is reserved for missing values."""
    edges = nn_bin_edges[name]
    if edges is None:
        codes = values.fillna(-1).clip(upper=nn_int_max[name]).to_numpy() + 1
    else:
        codes = np.where(values.isna(), 0, np.digitize(values.fillna(0), edges) + 1)
    return codes.astype("int64")


NN_EMBED_COLS = [f"emb_{name}" for name in NN_EMBED] + [f"bin_{name}" for name in nn_bin_edges]
NN_CARDINALITY = ([len(train[name].cat.categories) if isinstance(train[name].dtype, pd.CategoricalDtype)
                   else int(max(train[name].max(), test[name].max())) + 1 for name in NN_EMBED]
                  + [nn_int_max[name] + 2 if edges is None else len(edges) + 2 for name, edges in nn_bin_edges.items()])


def prepare_nn(frame):
    """Scaled numeric columns, a missing-delay flag, and integer codes for the embedded and binned columns."""
    out = ((nn_numeric(frame).fillna(nn_fill) - nn_mean) / nn_std).astype("float32").add_prefix("num_")
    out["num_arrival_missing"] = frame["Arrival Delay in Minutes"].isna().astype("float32")
    for name in NN_EMBED:
        values = frame[name].cat.codes if isinstance(frame[name].dtype, pd.CategoricalDtype) else frame[name]
        out[f"emb_{name}"] = values.astype("int64")
    for name in nn_bin_edges:
        out[f"bin_{name}"] = bin_codes(frame[name], name)
    return out


class TabularMLP(nn.Module):
    """Embeddings for the categorical and binned columns, concatenated with the numeric columns, then an MLP."""

    def __init__(self, cardinalities, n_numeric):
        super().__init__()
        dims = [CFG.NN_EMB_DIM if size <= 10 else CFG.NN_EMB_DIM_LARGE for size in cardinalities]
        self.embeddings = nn.ModuleList([nn.Embedding(size, dim) for size, dim in zip(cardinalities, dims)])
        width, layers = sum(dims) + n_numeric, []
        for hidden in CFG.NN_HIDDEN:
            layers += [nn.Linear(width, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(CFG.NN_DROPOUT)]
            width = hidden
        self.mlp = nn.Sequential(*layers, nn.Linear(width, 1))

    def forward(self, codes, numeric):
        embedded = [layer(codes[:, i]) for i, layer in enumerate(self.embeddings)]
        return self.mlp(torch.cat(embedded + [numeric], dim=1)).squeeze(1)


def nn_tensors(frame, device):
    """Embedding codes and numeric columns of a prepared frame, as tensors on the device."""
    numeric_cols = [name for name in frame.columns if name.startswith("num_")]
    return (torch.tensor(frame[NN_EMBED_COLS].to_numpy(), device=device),
            torch.tensor(frame[numeric_cols].to_numpy(), device=device))


@torch.no_grad()
def predict_mlp(model, inputs):
    """Probabilities for prepared inputs, in large batches."""
    model.eval()
    codes, numeric = inputs
    out = [torch.sigmoid(model(codes[i:i + 65536], numeric[i:i + 65536])) for i in range(0, len(codes), 65536)]
    return torch.cat(out).float().cpu().numpy()


def train_mlp(train_in, target, seed, valid_in=None, y_valid=None, stop_after=None):
    """Train one MLP on the one-cycle schedule over NN_EPOCHS.

    With validation data, keep the epoch with the best validation score. Without it (the refit), stop after
    `stop_after` epochs. Returns the model and the epoch it was taken from.
    """
    rows, sign = len(target), 1 if CFG.METRIC == "auc" else -1
    steps = math.ceil(rows / CFG.NN_BATCH)
    torch.manual_seed(CFG.SEED + seed)
    model = TabularMLP(NN_CARDINALITY, train_in[1].shape[1]).to(target.device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.NN_LR, weight_decay=CFG.NN_WEIGHT_DECAY)
    schedule = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=CFG.NN_LR, total_steps=CFG.NN_EPOCHS * steps)
    best, best_state, best_epoch = -np.inf, None, 0
    for epoch in range(stop_after or CFG.NN_EPOCHS):
        model.train()
        order = torch.randperm(rows, device=target.device)
        for start in range(0, rows, CFG.NN_BATCH):
            batch = order[start:start + CFG.NN_BATCH]
            if len(batch) > 1:  # BatchNorm needs at least two rows.
                loss = F.binary_cross_entropy_with_logits(model(train_in[0][batch], train_in[1][batch]), target[batch])
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
            schedule.step()
        if valid_in is not None:
            epoch_score = sign * score(y_valid, predict_mlp(model, valid_in))
            if epoch_score > best:  # Keep the best epoch, like early stopping in the trees.
                best, best_state, best_epoch = epoch_score, copy.deepcopy(model.state_dict()), epoch + 1
    if valid_in is None:
        return model, stop_after or CFG.NN_EPOCHS
    model.load_state_dict(best_state)
    return model, best_epoch


def fit_nn(x_train, y_train, x_valid, y_valid, x_test):
    """Train the MLP once per seed (CFG.NN_DEVICE) and return seed-averaged validation and test probabilities."""
    device = torch.device(CFG.NN_DEVICE)
    train_in, valid_in, test_in = (nn_tensors(frame, device) for frame in (x_train, x_valid, x_test))
    target = torch.tensor(y_train, dtype=torch.float32, device=device)
    valid_pred, test_pred, best_epochs = 0.0, 0.0, []
    for seed in range(CFG.NN_SEEDS):
        model, best_epoch = train_mlp(train_in, target, seed, valid_in, y_valid)
        best_epochs.append(best_epoch)
        fit_nn.best_epochs.append(best_epoch)  # All folds, for the refit.
        valid_pred += predict_mlp(model, valid_in) / CFG.NN_SEEDS
        test_pred += predict_mlp(model, test_in) / CFG.NN_SEEDS
    print(f"  NN best epochs per seed (of {CFG.NN_EPOCHS}): {best_epochs}")
    return valid_pred, test_pred


fit_nn.best_epochs = []


## 6. Train or reuse each model

Models listed in `CFG.REUSE` load their saved out-of-fold and test predictions from `preds/`. The others train across the fixed folds.

In [ ]:
def cross_validate(fit_model, name, prepare=lambda frame: frame):
    """Train all folds, fill out-of-fold scores, and average test scores."""
    x, x_test = prepare(train[FEATURES]), prepare(test[FEATURES])  # Identical columns for every model, prepared once.
    x_extra, y_extra = (prepare(original[FEATURES]), original[CFG.TARGET].to_numpy()) if original is not None else (None, None)
    oof, predicted_test = np.zeros(len(train)), np.zeros(len(test))  # Held-out and averaged test predictions.
    started = time.time()
    for number in range(CFG.N_FOLDS):  # Fit once for each validation fold.
        train_idx, valid_idx = np.flatnonzero(fold_id != number), np.flatnonzero(fold_id == number)
        x_fit, y_fit = x.iloc[train_idx], y[train_idx]
        if x_extra is not None:  # Original rows only ever train; validation stays competition data.
            x_fit, y_fit = pd.concat([x_fit, x_extra], ignore_index=True), np.concatenate([y_fit, y_extra])
        valid_pred, test_pred = fit_model(x_fit, y_fit, x.iloc[valid_idx], y[valid_idx], x_test)
        oof[valid_idx] = valid_pred  # Put held-out scores in their original positions.
        predicted_test += test_pred / CFG.N_FOLDS  # Average test scores across folds.
        print(f"{name} fold {number + 1}: {CFG.METRIC}={score(y[valid_idx], valid_pred):.5f}")
        gc.collect()  # Reclaim objects no longer held by the fold.
    run_minutes[name] = (time.time() - started) / 60  # Keep training time for the experiment log.
    print(f"{name} OOF {CFG.METRIC}={score(y, oof):.5f}; {run_minutes[name]:.1f} min")
    return oof, predicted_test


def reuse(name, source):
    """Load one model's saved out-of-fold and test predictions from an earlier experiment."""
    paths = [PRED_DIR / f"{source}_{name}_{part}.npy" for part in ("oof", "test")]
    missing = [path.name for path in paths if not path.is_file()]
    assert not missing, f"Missing {missing}: check preds/, or set CFG.REUSE['{name}'] = None to retrain."
    oof, predicted_test = (np.load(path) for path in paths)
    assert len(oof) == len(train) and len(predicted_test) == len(test), f"{source} predictions do not match the data."
    print(f"{name} reused from {source}: {CFG.METRIC}={score(y, oof):.5f}")
    return oof, predicted_test


TREE_DEVICE = "cuda" if CFG.GPU else "cpu"  # XGBoost and CatBoost
MODELS = {  # name: (enabled, fit function, one-off input preparation, device it trains on)
    "LightGBM": (CFG.RUN_LIGHTGBM, fit_lightgbm, lambda frame: frame, "cpu"),
    "XGBoost": (CFG.RUN_XGBOOST, fit_xgboost, lambda frame: frame, TREE_DEVICE),
    "CatBoost": (CFG.RUN_CATBOOST, fit_catboost, catboost_inputs, TREE_DEVICE),
    "CatBoostFast": (CFG.RUN_CATBOOST_FAST, fit_catboost_fast, catboost_inputs, TREE_DEVICE),
    "CatBoostOrig": (CFG.RUN_CATBOOST_ORIG, fit_catboost_orig, catboost_orig_inputs, TREE_DEVICE),
    "NN": (CFG.RUN_NN, fit_nn, prepare_nn, CFG.NN_DEVICE),
}
run_minutes, oof_predictions, test_predictions, devices = {}, {}, {}, {}  # Per-model time, scores, and hardware.
for name, (enabled, fit_model, prepare, device) in MODELS.items():
    if not enabled:
        continue
    if CFG.REUSE.get(name):
        oof_predictions[name], test_predictions[name] = reuse(name, CFG.REUSE[name])
        run_minutes[name], devices[name] = 0.0, f"reused {CFG.REUSE[name]}"
    else:
        oof_predictions[name], test_predictions[name] = cross_validate(fit_model, name, prepare)
        devices[name] = device

if hasattr(fit_lightgbm, "last_model"):  # Only when LightGBM trained in this run.
    importance = pd.Series(fit_lightgbm.last_model.booster_.feature_importance("gain"), index=FEATURES)
    print((importance / importance.sum()).sort_values(ascending=False).head(15).round(4).to_string())

## 7. Blend and write the submission

Blend weights for all models are chosen together on out-of-fold predictions by hill climbing (ensemble selection).

In [ ]:
results = {name: score(y, values) for name, values in oof_predictions.items()}  # Score each model once.
sign = 1 if CFG.METRIC == "auc" else -1  # Hill climbing maximises sign * score.


def hill_climb(oofs, steps=50):
    """Ensemble selection: repeatedly add (with replacement) the model that most improves the OOF score."""
    counts = dict.fromkeys(oofs, 0)
    first = max(oofs, key=lambda name: sign * score(y, oofs[name]))  # Start from the best single model.
    counts[first] = 1
    blend, best, best_counts = oofs[first].copy(), sign * score(y, oofs[first]), dict(counts)
    for step in range(1, steps):
        trials = {name: (blend * step + oofs[name]) / (step + 1) for name in oofs}  # Add one more vote to each model.
        trial_scores = {name: sign * score(y, values) for name, values in trials.items()}
        pick = max(trial_scores, key=trial_scores.get)
        counts[pick] += 1
        blend = trials[pick]
        if trial_scores[pick] > best:  # Keep the best combination found so far.
            best, best_counts = trial_scores[pick], dict(counts)
    total = sum(best_counts.values())
    return {name: count / total for name, count in best_counts.items()}


if len(oof_predictions) > 1:  # Choose all blend weights together.
    blend_weights = hill_climb(oof_predictions)
    final_oof = sum(weight * oof_predictions[name] for name, weight in blend_weights.items())
    final_test = sum(weight * test_predictions[name] for name, weight in blend_weights.items())
    results["Blend"] = score(y, final_oof)
else:  # A single model needs no blend search.
    blend_weights = {name: 1.0 for name in oof_predictions}
    final_oof, final_test = next(iter(oof_predictions.values())), next(iter(test_predictions.values()))
blend_note = ", ".join(f"{name} {weight:.2f}" for name, weight in blend_weights.items())
print("Blend weights:", blend_note)
print(pd.Series(results).round(5).to_string())

prediction_column = sample.columns[1]  # The competition's probability column name.
submission = sample[[CFG.ID]].copy()  # Preserve sample submission row order.
submission[prediction_column] = submission[CFG.ID].map(pd.Series(final_test, index=test[CFG.ID].to_numpy()))
assert submission[prediction_column].notna().all(), "Some test IDs lack predictions."
assert submission[prediction_column].between(0, 1).all(), "Predictions must be probabilities."
assert list(submission.columns) == list(sample.columns), "Submission columns changed."
if CFG.UPDATE_SUBMISSION:
    submission.to_csv(SUB_DIR / "submission.csv", index=False)  # The file to upload.
submission.to_csv(SUB_DIR / f"submission_{CFG.EXPERIMENT}.csv", index=False)  # A copy per experiment.
print(f"Saved submission.csv and submission_{CFG.EXPERIMENT}.csv with {len(submission):,} rows")

## 8. Full-data refit

Runs when `CFG.REFIT` is on. Each model with blend weight is retrained on all training rows, using the blend weights chosen above. LightGBM uses the run's mean best iteration × `REFIT_TREE_FACTOR`. The neural net keeps its schedule and stops at the run's median best epoch. For a reused model, `REFIT_LGB_TREES` and `REFIT_NN_EPOCHS` are used instead. The refit's test predictions replace `submission.csv` and `submission_<EXPERIMENT>.csv`. The fold-averaged version is kept as `submission_<EXPERIMENT>_foldavg.csv`.

CV cannot score a refit, so the public leaderboard is the check. Compare it with the fold-averaged submission's score.

In [ ]:
if CFG.REFIT:
    assert original is None, "The refit does not add the original dataset yet."
    # Trained in this run: size the refit from this run's CV. Reused: fall back to the CFG values.
    refit_trees = (int(round(np.mean(fit_lightgbm.best_iterations) * CFG.REFIT_TREE_FACTOR)) if fit_lightgbm.best_iterations
                   else CFG.REFIT_LGB_TREES)
    refit_epochs = int(np.median(fit_nn.best_epochs)) if fit_nn.best_epochs else CFG.REFIT_NN_EPOCHS
    refit_cb_trees = {name: int(round(np.mean(fitter.best_iterations) * CFG.REFIT_TREE_FACTOR))
                      for name, fitter in CATBOOST_FITTERS.items() if fitter.best_iterations}
    print(f"Refit sizes: LightGBM {refit_trees} trees, NN {refit_epochs} epochs, CatBoost {refit_cb_trees}")
    PRED_DIR.mkdir(exist_ok=True)
    refit_test, refit_notes = {}, []
    for name, weight in blend_weights.items():
        if not weight:
            continue
        started = time.time()
        source = CFG.REUSE.get(name)
        saved = PRED_DIR / f"{source}_{name}_refit_test.npy" if source else None
        if saved is not None and saved.is_file():  # Reused model: its experiment already refit it.
            prediction = np.load(saved)
            refit_notes.append(f"{name} refit from {source}")
        elif name == "LightGBM":
            prediction = 0.0
            for seed in range(CFG.LGB_SEEDS):
                model = make_lightgbm(seed, refit_trees).fit(train[FEATURES], y)
                prediction += model.predict_proba(test[FEATURES])[:, 1] / CFG.LGB_SEEDS
            refit_notes.append(f"LightGBM {CFG.LGB_SEEDS}x{refit_trees} trees")
        elif name == "NN":
            device = torch.device(CFG.NN_DEVICE)
            full_in, test_in = nn_tensors(prepare_nn(train[FEATURES]), device), nn_tensors(prepare_nn(test[FEATURES]), device)
            target = torch.tensor(y, dtype=torch.float32, device=device)
            prediction = 0.0
            for seed in range(CFG.NN_SEEDS):
                model, _ = train_mlp(full_in, target, seed, stop_after=refit_epochs)
                prediction += predict_mlp(model, test_in) / CFG.NN_SEEDS
            refit_notes.append(f"NN {CFG.NN_SEEDS}x{refit_epochs} epochs")
        elif name in refit_cb_trees:
            fitter, prediction = CATBOOST_FITTERS[name], 0.0
            x_full, y_full = fitter.prepare(train[FEATURES]), y
            if fitter.extra is not None:  # CatBoostOrig: the original rows join the refit too.
                x_full, y_full = pd.concat([x_full, fitter.extra[0]], ignore_index=True), np.concatenate([y, fitter.extra[1]])
            for seed in fitter.seeds:
                model = make_catboost(refit_cb_trees[name], early_stopping=False, learning_rate=fitter.learning_rate, seed=seed)
                model.fit(Pool(x_full, y_full, cat_features=fitter.cat_features))
                prediction += model.predict_proba(fitter.prepare(test[FEATURES]))[:, 1] / len(fitter.seeds)
            refit_notes.append(f"{name} {len(fitter.seeds)}x{refit_cb_trees[name]} trees")
        else:  # No refit for this model, so its fold-averaged test predictions are used.
            prediction = test_predictions[name]
            refit_notes.append(f"{name} fold-averaged")
        refit_test[name] = prediction
        np.save(PRED_DIR / f"{CFG.EXPERIMENT}_{name}_refit_test.npy", prediction)
        print(f"{name} refit: {(time.time() - started) / 60:.1f} min | "
              f"correlation with its fold-averaged test predictions {np.corrcoef(prediction, test_predictions[name])[0, 1]:.5f}")
    refit_final = sum(weight * refit_test[name] for name, weight in blend_weights.items() if weight)
    np.save(PRED_DIR / f"{CFG.EXPERIMENT}_Blend_refit_test.npy", refit_final)
    print(f"Blend refit vs fold-averaged: correlation {np.corrcoef(refit_final, final_test)[0, 1]:.5f}, "
          f"mean {refit_final.mean():.4f} vs {final_test.mean():.4f}")

    submission.to_csv(SUB_DIR / f"submission_{CFG.EXPERIMENT}_foldavg.csv", index=False)  # Keep the fold-averaged version.
    submission[prediction_column] = submission[CFG.ID].map(pd.Series(refit_final, index=test[CFG.ID].to_numpy()))
    assert submission[prediction_column].notna().all() and submission[prediction_column].between(0, 1).all()
    if CFG.UPDATE_SUBMISSION:
        submission.to_csv(SUB_DIR / "submission.csv", index=False)  # The file to upload.
    submission.to_csv(SUB_DIR / f"submission_{CFG.EXPERIMENT}.csv", index=False)
    blend_note += " | refit: " + ", ".join(refit_notes)
    print(f"Saved refit submission.csv and submission_{CFG.EXPERIMENT}.csv; fold-averaged copy in submission_{CFG.EXPERIMENT}_foldavg.csv")

## 9. Record the experiment

Appends one row per model to `experiments.csv` and saves OOF and test predictions in `preds/`. The `device` column says whether each model trained on `cpu`, `mps` or `cuda`, or was reused. `delta_vs_baseline` and `folds_improved` compare with the same model in `CFG.BASELINE` on the same folds. Treat a gain as real only if it is at least +0.0002 and improves at least 4 of 5 folds. After uploading, write the public score into `public_lb` on the `Blend` row.

In [ ]:
def fold_scores(oof):
    """Score each fixed fold separately."""
    return [score(y[fold_id == k], oof[fold_id == k]) for k in range(CFG.N_FOLDS)]


def record_experiment():
    """Append this run to experiments.csv and save its predictions in preds/."""
    log_path, pred_dir = OUT_DIR / "experiments.csv", OUT_DIR / "preds"
    pred_dir.mkdir(exist_ok=True)
    suffix = "-fast" if CFG.FAST else ""  # FAST rows only compare with other FAST rows.
    runs = {**oof_predictions, "Blend": final_oof} if "Blend" in results else dict(oof_predictions)
    tests = {**test_predictions, "Blend": final_test}
    log = (pd.read_csv(log_path).rename(columns={"oof_auc": "oof_score", "fold_aucs": "fold_scores"})
           if log_path.is_file() else pd.DataFrame())  # Older logs used AUC-specific names.
    rows = []
    for name, oof in runs.items():
        model, folds = name + suffix, fold_scores(oof)
        base = log[(log.experiment == CFG.BASELINE) & (log.model == model)] if len(log) else log
        delta, improved = np.nan, np.nan
        if len(base) and CFG.EXPERIMENT != CFG.BASELINE:
            delta = score(y, oof) - base.oof_score.iloc[0]
            improved = sum(sign * (round(f, 5) - b) > 0 for f, b in zip(folds, [float(v) for v in base.fold_scores.iloc[0].split()]))
        previous = log[(log.experiment == CFG.EXPERIMENT) & (log.model == model)] if len(log) else log
        rows.append({
            "experiment": CFG.EXPERIMENT, "model": model, "n_features": len(FEATURES),
            "fold_scores": " ".join(f"{f:.5f}" for f in folds), "oof_score": round(score(y, oof), 5),
            "delta_vs_baseline": round(delta, 5), "folds_improved": improved,
            "minutes": round(run_minutes.get(name, 0.0), 1), "fold_seed": CFG.SEED,
            "notes": blend_note if name == "Blend" else "", "device": devices.get(name, ""),
            "public_lb": previous.public_lb.iloc[0] if len(previous) else np.nan,
            "logged_at": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
        })
        np.save(pred_dir / f"{CFG.EXPERIMENT}_{model}_oof.npy", oof)
        np.save(pred_dir / f"{CFG.EXPERIMENT}_{model}_test.npy", tests[name])
    if len(log):  # Replace earlier rows from a rerun of the same experiment.
        log = log[~((log.experiment == CFG.EXPERIMENT) & log.model.isin([r["model"] for r in rows]))]
    log = pd.concat([log, pd.DataFrame(rows)], ignore_index=True)
    log.to_csv(log_path, index=False)
    return log


if CFG.DEBUG:  # Debug runs use a sample and would pollute the log.
    print("DEBUG run: experiment not recorded.")
else:
    experiment_log = record_experiment()
    print(experiment_log[experiment_log.experiment.isin([CFG.BASELINE, CFG.EXPERIMENT])].drop(columns=["logged_at"]).to_string(index=False))